In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
from scipy import integrate
import glob
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import pandas as pd
import matplotlib.dates as mdates
import pandas as pd


In [ ]:
# predefine what to load
def preprocessor(ds, dataset_name):
    if dataset_name == "waccm":
        # Preprocessing for the first dataset
        # ds = ds.sel(lat=slice(-70, -40))
        ds = ds[['U','UAP','UBP','UTEND_PHYSTOT','UTEND_CORE','V','T','OMEGA']]  # Select only the 'U' variable
    else:
        raise ValueError(f"Unknown dataset name: {dataset_name}")
    return ds

In [ ]:
# set up constants for TEM calculations
Omega_rot=7.29212e-5
a        = 6.371e6   # Earth radius, m
g0       = 9.81      # gravity, m/s²
H        = 7000.     # scale height, m
p0       = 101325.   # reference pressure, Pa
R        = 287.  # J/kg/K, dry air gas constant
kappa    = R/1004.     # R/cp for dry air



In [ ]:
#print(poopypants)

In [ ]:
#ds = ds.isel(lev=slice(None, None, -1))
lat = np.array(ds.lat)
pre = np.array(ds.lev)
latrad = np.array((ds.lat/180.)*np.pi)
f=2.*Omega_rot*np.sin(latrad[:])

npre = pre.size
nlat = lat.size
nlon = ds.lon.size
ntime = ds.time.size

pre_pa      = pre * 100.0                        # hPa → Pa
p_ratio_3d  = (p0 / pre_pa)[np.newaxis, :, np.newaxis] ** kappa
# ------------------------------------------------------------------
# dt_seconds — derive from the time coordinate directly
# Use the median spacing so one bad gap doesn't break things.
# With uniform 6-hourly data this will simply be 21600.0 s.
# ------------------------------------------------------------------
# Convert xarray time coord to numpy datetime64, then to float seconds
time_np      = ds.time.values.astype('datetime64[s]').astype(np.float64)  # seconds since epoch
dt_seconds   = float(np.median(np.diff(time_np)))   # e.g. 21600.0 for 6-hourly
print(f"Detected timestep: {dt_seconds / 3600:.1f} hours")


In [ ]:
"""
TEM (Transformed Eulerian Mean) Diagnostics — minimum memory version
Equations follow Appendix A of DynVarMIP (Gerber & Manzini 2016,
Geosci. Model Dev., 9, 3413-3425, including the December 2016 corrigendum).

FIX (vs. previous version)
---------------------------
The paper's Appendix A is formulated entirely in PRESSURE coordinates.
The vertical velocity is omega = dp/dt  [Pa s-1]  (CMIP6/CMOR variable "wap").
The previous version used the name  w / wzm  and carried comments that
suggested a conversion from omega to the geometric vertical velocity w [m s-1].
No such conversion is performed or required.  The variable has been renamed
om / omzm / uomzm throughout to match the paper's notation.

The log-pressure scaling in Step 10 (Eq A16) converts the TEM residual
vertical velocity from pressure coordinates to the log-pressure coordinate:
    w-hat_tem = -(H / p) * omega_tem          (Eq A16)
This is a *diagnostic re-expression* for display purposes, not a conversion
of the input data.

Corrigendum note (published 21 Dec 2016)
-----------------------------------------
Eq A15 in the original paper incorrectly equated the EP-flux divergence in
log-pressure coordinates to the zonal-wind tendency.  The corrected A15 is:
    du/dt|_{nabla.F}  =  nabla.F / (a cos phi)
with nabla.F the EP-flux divergence in pressure coordinates (Eq A5).
The implementation below already reflects the corrected formulation.

Memory strategy
---------------
The 4D input arrays U, V, omega, T are never all loaded at once.
A single longitude slice is read at a time; zonal means and eddy
flux sums are accumulated on the fly.  The caller passes a loader
function rather than pre-loaded arrays, so data can come straight
from NetCDF (or any source) one slice at a time.

Interface
---------
Provide a callable  load_slice(var, k)  that returns the 3D array
(ntime, npre, nlat) for variable var in {'U','V','OM','T'} at
longitude index k.  'OM' is omega = dp/dt [Pa s-1].
Example using netCDF4:

    import netCDF4 as nc
    ds = nc.Dataset("model_output.nc")
    def load_slice(var, k):
        cmor = {'U': 'ua', 'V': 'va', 'OM': 'wap', 'T': 'ta'}
        return ds[cmor[var]][:, :, :, k]     # (ntime, npre, nlat)

Parameters (scalars / 1-D arrays, defined before calling this code)
----------
ntime, npre, nlat, nlon : int  -- grid dimensions
pre   : np.ndarray, shape (npre,)   -- pressure levels, hPa, ascending (TOA to surface)
lat   : np.ndarray, shape (nlat,)   -- latitude, degrees
a     : float  -- Earth radius, m          (e.g. 6.371e6)
g0    : float  -- gravity, m/s^2           (e.g. 9.81)
H     : float  -- scale height, m          (e.g. 7000.)
p0    : float  -- reference pressure, Pa   (e.g. 101325.)
kappa : float  -- R/cp                     (e.g. 2./7.)
f     : np.ndarray, shape (nlat,)  -- Coriolis parameter, s-1
load_slice : callable(var: str, k: int) -> np.ndarray (ntime, npre, nlat)
"""

# ------------------------------------------------------------------
# Sanity check
# ------------------------------------------------------------------
assert np.all(np.diff(pre) > 0), "pre must be ascending (TOA to surface)"

In [ ]:

print( pre.shape )
pre_4d = np.tile( pre[None,:,None,None] ,  (ntime,1,nlat,nlon) )
print( pre_4d.shape )
print(p0)

In [ ]:
%%time


p_ratio_4d= np.tile( p_ratio_3d[None,:,:,:], (ntime,1,nlat,nlon) )

omega = ds.OMEGA.values
print('got omega' )
u= ds.U.values
print('got U' )
v= ds.V.values
print('got V' )
te= ds.T.values
print('got T' )

th=te * ( (1_000. / pre_4d) ** kappa )


In [ ]:
#ze = np.tile(ze[None, :, :, :], (nt, 1, 1, 1) )
p_ratio_4d.shape


In [ ]:

uzm=np.mean( u,axis=3 )
vzm=np.mean( v,axis=3 )
omzm=np.mean( omega,axis=3 )
thzm=np.mean( th,axis=3 )



uvzm  = np.mean( u*v,axis=3 ) - uzm * vzm     # [u*v*]   Eq A2
uomzm = np.mean( u*omega,axis=3 ) - uzm * omzm    # [u*omega*]   Eq A3  (omega [Pa s-1], not w [m s-1])
vthzm = np.mean( v*th,axis=3) - vzm * thzm    # [v*theta*]   Eq A4


In [ ]:
print(pre)

In [ ]:
zo=-7.*np.log( pre / 1_000. )
plt.contourf( lat,zo, thzm[0,:,:] )
plt.colorbar()

In [ ]:
# ------------------------------------------------------------------
# Cached coordinate arrays
# ------------------------------------------------------------------
pre_pa  = pre * 100.0                                         # hPa -> Pa, (npre,)
latrad  = np.deg2rad(lat)                                     # (nlat,)
cos_lat = np.cos(latrad).clip(min=1e-10)                      # guard poles, (nlat,)

# Broadcast shapes for (ntime, npre, nlat) output fields
acos     = a * cos_lat[np.newaxis, np.newaxis, :]             # (1, 1, nlat)
prearray = pre_pa[np.newaxis, :, np.newaxis]                  # (1, npre, 1)
farray   = f[np.newaxis, np.newaxis, :]                       # (1, 1, nlat)

# theta conversion factor per pressure level, shape (1, npre, 1)
p_ratio_3d = (p0 / pre_pa)[np.newaxis, :, np.newaxis] ** kappa


print( ntime, npre, nlat, nlon ) 

In [ ]:

pre_pa  = pre * 100.0                                         # hPa -> Pa, (npre,)
latrad  = np.deg2rad(lat)                                     # (nlat,)
cos_lat = np.cos(latrad).clip(min=1e-10)                      # guard poles, (nlat,)



# ------------------------------------------------------------------
# Step 2 -- Latitudinal gradient of U  (Eq A2/A3 component)
# (1 / a*cos phi) * d([U]*cos phi)/d phi
# ------------------------------------------------------------------
dudphi = (1.0 / acos) * np.gradient(
    uzm * cos_lat[np.newaxis, np.newaxis, :], latrad, axis=2
)

# ------------------------------------------------------------------
# Step 3 -- Vertical gradients
# ------------------------------------------------------------------
dthdp = np.gradient(thzm, pre_pa, axis=1)   # d[theta]/dp
dudp  = np.gradient(uzm,  pre_pa, axis=1)   # d[U]/dp

In [ ]:
# ------------------------------------------------------------------
# Step 4 -- Eddy stream function psi* and its gradients  (Eq A4)
#
# psi* = [v*theta*] / (d[theta]/dp)
# Both numerator and denominator are in pressure coordinates.
# ------------------------------------------------------------------
stability_threshold = 1e-6   # K/Pa -- adjust to your data if needed
psieddy = np.where(
    np.abs(dthdp) > stability_threshold,
    vthzm / dthdp,
    np.nan
)   # Eq A4

dpsidp = np.gradient(psieddy, pre_pa, axis=1)

psicos = psieddy * cos_lat[np.newaxis, np.newaxis, :]
dpsidy = (1.0 / acos) * np.gradient(psicos, latrad, axis=2)

In [ ]:
print( " Done upto Step 5 " )

# ------------------------------------------------------------------
# Step 5 -- TEM residual velocities  (Eq A6, A7)
#
# v* = [v] - d psi*/dp                              Eq A6
# omega* = [omega] + (1/a cos phi) d(psi* cos phi)/d phi   Eq A7
#
# Both velocities remain in pressure coordinates here.
# omega*_tem has units Pa s-1.
# ------------------------------------------------------------------
omtem = omzm + dpsidy    # omega*_tem [Pa s-1]   Eq A7
vtem  = vzm  - dpsidp    # v*_tem [m s-1]        Eq A6

print( " Done upto Step 6 " )

# ------------------------------------------------------------------
# Step 6 -- Zonal wind tendencies  (Eq A9, A10)
# ------------------------------------------------------------------
utendvtem = vtem  * (farray - dudphi)   # Eq A9
utendwtem = -1.0  * omtem * dudp        # Eq A10  (omega*_tem * d[u]/dp)

print( " Done upto Step 7 " )

# ------------------------------------------------------------------
# Step 7 -- E-P flux vectors  (Eq A2, A3)
#
# F^(phi) = a cos phi ( d[u]/dp * psi* - [u*v*] )              Eq A2
# F^(p)   = a cos phi ( (f - d[u]cos phi/a cos phi d phi) psi* - [u*omega*] )  Eq A3
#
# [u*omega*] uses omega [Pa s-1] directly -- no conversion needed.
# ------------------------------------------------------------------
epfy = acos * (dudp * psieddy - uvzm )               # Eq A2
epfz = acos * ((farray - dudphi) * psieddy - uomzm)  # Eq A3

print( " Done upto Step 8 " )

In [ ]:

print( " Done upto Step 8 " )
# ------------------------------------------------------------------
# Step 8 -- E-P flux divergence and zonal wind tendency  (Eq A5, A15)
#
# nabla.F = (1/a cos phi) d(F^(phi) cos phi)/d phi + dF^(p)/dp   Eq A5
#
# Corrigendum (Dec 2016) correction to Eq A15:
#   utendepfd = nabla.F / (a cos phi)
# ------------------------------------------------------------------
depfydphi = (1.0 / acos) * np.gradient(
    epfy * cos_lat[np.newaxis, np.newaxis, :], latrad, axis=2
)
depfzdp   = np.gradient(epfz, pre_pa, axis=1)

utendepfd = (1.0 / acos) * (depfydphi + depfzdp)   # Eq A5 + corrected A15

print( " Done upto Step 9 " )

# ------------------------------------------------------------------
# Step 9 -- TEM stream function Psi  (Eq A8)
#
# Psi = (2 pi a / g0) cos phi [ int_0^p [v] dp' - psi* ]
# ------------------------------------------------------------------
topvzm      = np.zeros([ntime, 1, nlat])
vzmwithzero = np.concatenate((topvzm, vzm), axis=1)
prewithzero = np.concatenate(([0.0], pre)) * 100.0

intv = integrate.cumulative_trapezoid(vzmwithzero, prewithzero, axis=1)

psitem = (2.0 * np.pi * a / g0) * acos * (intv - psieddy)  # Eq A8

In [ ]:
print(epfy.shape)
print(pre_4d.shape)
print(H,p0)

prearray=np.mean( pre_4d*100. , axis=3 )

In [ ]:
print( " Done upto Step 10 " )

# ------------------------------------------------------------------
# Step 10 -- Log-pressure coordinate scaling  (Eq A13, A14, A16)
#
# F-hat^(phi) = (p/p0) F^(phi)                     Eq A13
# F-hat^(z)   = -(H/p0) F^(p)                      Eq A14
# w-hat_tem   = -(H/p)  omega*_tem                  Eq A16
#
# Eq A16 converts the TEM residual vertical velocity from the
# pressure-coordinate quantity omega*_tem [Pa s-1] to the log-pressure
# vertical velocity w-hat_tem [m s-1] for display in z* coordinates.
# This is a diagnostic re-expression; the input OM was never w [m s-1].
# ------------------------------------------------------------------
epfy  =        epfy  * prearray / p0        # Eq A13
epfz  = -1.0 * (H / p0) * epfz             # Eq A14
wtem  = -1.0 * (H / prearray) * omtem      # Eq A16  (renamed omtem -> wtem for output)

# ------------------------------------------------------------------
# Output summary -- all arrays shape (ntime, npre, nlat)
# ------------------------------------------------------------------
# uzm, vzm, omzm, thzm -- zonal mean fields
#   omzm : [omega] = zonal mean omega [Pa s-1]
# uvzm, uomzm, vthzm   -- eddy flux zonal means
#   uomzm : [u*omega*]  [m s-1 * Pa s-1]
# psieddy              -- eddy stream function psi*                 (Eq A4)
# vtem                 -- TEM meridional velocity [m s-1]           (Eq A6)
# wtem                 -- TEM log-pressure vertical velocity [m s-1] (Eq A16)
# epfy, epfz           -- E-P flux components (log-pressure scaled) (Eq A13, A14)
# utendvtem            -- zonal wind tendency from vtem             (Eq A9)
# utendwtem            -- zonal wind tendency from omega*_tem       (Eq A10)
# utendepfd            -- zonal wind tendency from EP flux divergence (Eq A5 + corr. A15)
# psitem               -- TEM mass stream function Psi              (Eq A8)

In [ ]:

import matplotlib.ticker as ticker

# ── Setup ────────────────────────────────────────────────────────────────────
# All data arrays have shape (time, lev, lat)
# Units: m/s/day  (raw utendvtem/utendwtem/utendepfd are m/s²; multiplying by
#                  seconds_per_day converts to m/s/day)

seconds_per_day = 86400

# Mask exact pole points (lat = ±90°) where 1/cosφ → ∞ in utendepfd.
_pole_mask = (lat == 90.0) | (lat == -90.0)
def _mask_poles(arr):
    """Set ±90° lat columns to NaN. arr shape: (time, lev, lat)."""
    out = arr.copy().astype(float)
    out[:, :, _pole_mask] = np.nan
    return out

utendepfd_m = _mask_poles(utendepfd * seconds_per_day)
utendvtem_m = utendvtem * seconds_per_day
utendwtem_m = utendwtem * seconds_per_day

utend_tem = utendvtem_m + utendwtem_m + utendepfd_m 
res       = utend_tem - ds_tend['UTEND_CORE'].mean(dim=['lon']).values * seconds_per_day

VARIABLES = {
    "utendvtem":   utendvtem_m,
    "utendwtem":   utendwtem_m,
    "utendepfd":   utendepfd_m,
    "utend_tem":   utend_tem,
    "UTEND_CORE":   ds_tend["UTEND_CORE"].mean(dim=["lon"]).values * seconds_per_day,
    "res":         res,
}

TITLES = {
    "utendvtem":  r"$\bar{v}^*$ tendency  (Eq. A9)",
    "utendwtem":  r"$\bar{\omega}^*$ tendency  (Eq. A10)",
    "utendepfd":  "EPF divergence tendency",
    "utend_tem":  "Total TEM tendency",
    "UTEND_CORE":  "UTEND_CORE (zonal mean)",
    "res":        "Residual (TEM - TOT)",
}

LEV_LAT_VMAX = {
    "utendvtem":  1e1,
    "utendwtem":  1e0,
    "utendepfd":  1e1,
    "utend_tem":  1e1,
    "UTEND_TOT":  None,
    "res":        1e1,
}

LEV_TIME_VMAX = {
    "utendvtem":  None,
    "utendwtem":  1e0,
    "utendepfd":  1e1,
    "utend_tem":  1e1,
    "UTEND_TOT":  None,
    "res":        1e1,
}

LAT_MIN, LAT_MAX = -85, 85

lat       = ds_tend["lat"].values
lev       = ds_tend["lev"].values
try:
    time_vals = ds_tend["time"].values
except Exception:
    time_vals = None

CMAP      = "RdBu_r"
NLEVS     = 20
UNIT      = "m s-1 day-1"
LEV_TICKS = [100, 10, 1, 0.1]


def _sym_levels(data, n=NLEVS, vmax_override=None):
    vmax = vmax_override if vmax_override is not None else float(np.nanpercentile(np.abs(data), 98))
    if vmax == 0:
        vmax = 1e-8
    return np.linspace(-vmax, vmax, n + 1)

def _contourf_kwargs(data, vmax_override=None):
    return dict(levels=_sym_levels(data, vmax_override=vmax_override), cmap=CMAP, extend="both")

def _add_colorbar(fig, cf, ax):
    cb = fig.colorbar(cf, ax=ax, orientation="vertical", pad=0.02, fraction=0.046)
    cb.set_label(UNIT, fontsize=8)
    cb.ax.tick_params(labelsize=7)

def _format_lev_ax(ax, xlabel):
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel("Pressure (hPa)", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.set_yscale("log")
    ax.set_ylim(1000, 0.1)
    ax.set_yticks(LEV_TICKS)
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:g}"))

def _make_grid(n, ncols=3):
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), constrained_layout=True)
    return fig, np.array(axes).flatten()


def plot_lev_lat(variables, titles):
    fig, axes = _make_grid(len(variables))
    for ax, (key, arr) in zip(axes, variables.items()):
        data    = arr.mean(axis=0)
        vmax_ov = LEV_LAT_VMAX.get(key)
        cf      = ax.contourf(lat, lev, data, **_contourf_kwargs(data, vmax_override=vmax_ov))
        ax.contour(lat, lev, data, levels=cf.levels[::2], colors="k", linewidths=0.4, alpha=0.5)
        _add_colorbar(fig, cf, ax)
        _format_lev_ax(ax, "Latitude (deg)")
        ax.set_title(titles[key], fontsize=10)
    for ax in axes[len(variables):]:
        ax.set_visible(False)
    fig.suptitle("Lev x Lat  (time-mean)", fontsize=13, fontweight="bold")
    fig.savefig("plot1_lev_lat.png", dpi=150)
    plt.show()
    print("Saved -> plot1_lev_lat.png")


def plot_lev_profile(variables, titles):
    fig, ax = plt.subplots(figsize=(5, 7))
    colors  = plt.cm.tab10(np.linspace(0, 1, len(variables)))
    for (key, arr), color in zip(variables.items(), colors):
        profile = np.nanmean(arr, axis=(0, 2))
        ax.plot(profile, lev, label=titles[key], color=color, linewidth=1.8)
    ax.axvline(0, color="k", linewidth=0.8, linestyle="--")
    ax.set_xlabel(f"Tendency ({UNIT})", fontsize=10)
    ax.set_ylabel("Pressure (hPa)", fontsize=10)
    ax.set_yscale("log")
    ax.set_ylim(1000, 0.1)
    ax.set_yticks(LEV_TICKS)
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:g}"))
    ax.tick_params(labelsize=9)
    ax.legend(fontsize=8, loc="best")
    ax.set_title("Vertical profile\n(mean over time & lat)", fontsize=11, fontweight="bold")
    fig.tight_layout()
    fig.savefig("plot2_lev_profile.png", dpi=150)
    plt.show()
    print("Saved -> plot2_lev_profile.png")


def plot_lev_time(variables, titles):
    if time_vals is not None:
        try:
            import cftime
            t0 = time_vals[0]
            t_axis = np.array([
                cftime.date2num(t, units=f"days since {t0.year}-{t0.month:02d}-{t0.day:02d}",
                                calendar=t0.calendar)
                for t in time_vals
            ])
            t_label = f"Days since {t0.year}-{t0.month:02d}-{t0.day:02d}"
        except Exception:
            t_axis = np.array(time_vals, dtype="float64")
            t_label = "Time"
    else:
        t_axis = np.arange(next(iter(variables.values())).shape[0])
        t_label = "Time index"
    lat_mask = (lat >= LAT_MIN) & (lat <= LAT_MAX)
    fig, axes = _make_grid(len(variables))
    for ax, (key, arr) in zip(axes, variables.items()):
        data    = np.nanmean(arr[:, :, lat_mask], axis=2)
        vmax_ov = LEV_TIME_VMAX.get(key)
        cf      = ax.contourf(t_axis, lev, data.T, **_contourf_kwargs(data, vmax_override=vmax_ov))
        ax.contour(t_axis, lev, data.T, levels=cf.levels[::2], colors="k", linewidths=0.4, alpha=0.5)
        _add_colorbar(fig, cf, ax)
        _format_lev_ax(ax, t_label)
        ax.set_title(titles[key], fontsize=10)
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right", fontsize=7)
    for ax in axes[len(variables):]:
        ax.set_visible(False)
    fig.suptitle("Lev x Time  (lat-mean)", fontsize=13, fontweight="bold")
    fig.savefig("plot3_lev_time.png", dpi=150)
    plt.show()
    print("Saved -> plot3_lev_time.png")


plot_lev_lat(VARIABLES, TITLES)
plot_lev_profile(VARIABLES, TITLES)
plot_lev_time(VARIABLES, TITLES)


In [ ]:

import matplotlib.ticker as ticker

# ── Setup ────────────────────────────────────────────────────────────────────
# All data arrays have shape (time, lev, lat)
# Units: m/s/day  (raw utendvtem/utendwtem/utendepfd are m/s²; multiplying by
#                  seconds_per_day converts to m/s/day)

seconds_per_day = 86400

# Mask exact pole points (lat = ±90°) where 1/cosφ → ∞ in utendepfd.
_pole_mask = (lat == 90.0) | (lat == -90.0)
def _mask_poles(arr):
    """Set ±90° lat columns to NaN. arr shape: (time, lev, lat)."""
    out = arr.copy().astype(float)
    out[:, :, _pole_mask] = np.nan
    return out

utendepfd_m = _mask_poles(utendepfd * seconds_per_day)
utendvtem_m = utendvtem * seconds_per_day
utendwtem_m = utendwtem * seconds_per_day

utend_tem = utendvtem_m + utendwtem_m + utendepfd_m + ds_tend['UTEND_PHYSTOT'].mean(dim=['lon']).values * seconds_per_day
res       = utend_tem - ds_tend['UTEND_TOT'].mean(dim=['lon']).values * seconds_per_day

VARIABLES = {
    "utendvtem":   utendvtem_m,
    "utendwtem":   utendwtem_m,
    "utendepfd":   utendepfd_m,
    "utend_tem":   utend_tem,
    "UTEND_TOT":   ds_tend["UTEND_TOT"].mean(dim=["lon"]).values * seconds_per_day,
    "res":         res,
}

TITLES = {
    "utendvtem":  r"$\bar{v}^*$ tendency  (Eq. A9)",
    "utendwtem":  r"$\bar{\omega}^*$ tendency  (Eq. A10)",
    "utendepfd":  "EPF divergence tendency",
    "utend_tem":  "Total TEM tendency",
    "UTEND_TOT":  "UTEND_TOT (zonal mean)",
    "res":        "Residual (TEM - TOT)",
}

LEV_LAT_VMAX = {
    "utendvtem":  None,
    "utendwtem":  1e0,
    "utendepfd":  1e2,
    "utend_tem":  1e1,
    "UTEND_TOT":  None,
    "res":        1e1,
}

LEV_TIME_VMAX = {
    "utendvtem":  None,
    "utendwtem":  1e0,
    "utendepfd":  1e2,
    "utend_tem":  1e1,
    "UTEND_TOT":  None,
    "res":        1e1,
}

LAT_MIN, LAT_MAX = -85, 85

lat       = ds_tend["lat"].values
lev       = ds_tend["lev"].values
try:
    time_vals = ds_tend["time"].values
except Exception:
    time_vals = None

CMAP      = "RdBu_r"
NLEVS     = 20
UNIT      = "m s-1 day-1"
LEV_TICKS = [100, 10, 1, 0.1]


def _sym_levels(data, n=NLEVS, vmax_override=None):
    vmax = vmax_override if vmax_override is not None else float(np.nanpercentile(np.abs(data), 98))
    if vmax == 0:
        vmax = 1e-8
    return np.linspace(-vmax, vmax, n + 1)

def _contourf_kwargs(data, vmax_override=None):
    return dict(levels=_sym_levels(data, vmax_override=vmax_override), cmap=CMAP, extend="both")

def _add_colorbar(fig, cf, ax):
    cb = fig.colorbar(cf, ax=ax, orientation="vertical", pad=0.02, fraction=0.046)
    cb.set_label(UNIT, fontsize=8)
    cb.ax.tick_params(labelsize=7)

def _format_lev_ax(ax, xlabel):
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel("Pressure (hPa)", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.set_yscale("log")
    ax.set_ylim(1000, 0.1)
    ax.set_yticks(LEV_TICKS)
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:g}"))

def _make_grid(n, ncols=3):
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), constrained_layout=True)
    return fig, np.array(axes).flatten()


def plot_lev_lat(variables, titles):
    fig, axes = _make_grid(len(variables))
    for ax, (key, arr) in zip(axes, variables.items()):
        data    = arr.mean(axis=0)
        vmax_ov = LEV_LAT_VMAX.get(key)
        cf      = ax.contourf(lat, lev, data, **_contourf_kwargs(data, vmax_override=vmax_ov))
        ax.contour(lat, lev, data, levels=cf.levels[::2], colors="k", linewidths=0.4, alpha=0.5)
        _add_colorbar(fig, cf, ax)
        _format_lev_ax(ax, "Latitude (deg)")
        ax.set_title(titles[key], fontsize=10)
    for ax in axes[len(variables):]:
        ax.set_visible(False)
    fig.suptitle("Lev x Lat  (time-mean)", fontsize=13, fontweight="bold")
    fig.savefig("plot1_lev_lat.png", dpi=150)
    plt.show()
    print("Saved -> plot1_lev_lat.png")


def plot_lev_profile(variables, titles):
    fig, ax = plt.subplots(figsize=(5, 7))
    colors  = plt.cm.tab10(np.linspace(0, 1, len(variables)))
    for (key, arr), color in zip(variables.items(), colors):
        profile = np.nanmean(arr, axis=(0, 2))
        ax.plot(profile, lev, label=titles[key], color=color, linewidth=1.8)
    ax.axvline(0, color="k", linewidth=0.8, linestyle="--")
    ax.set_xlabel(f"Tendency ({UNIT})", fontsize=10)
    ax.set_ylabel("Pressure (hPa)", fontsize=10)
    ax.set_yscale("log")
    ax.set_ylim(1000, 0.1)
    ax.set_yticks(LEV_TICKS)
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:g}"))
    ax.tick_params(labelsize=9)
    ax.legend(fontsize=8, loc="best")
    ax.set_title("Vertical profile\n(mean over time & lat)", fontsize=11, fontweight="bold")
    fig.tight_layout()
    fig.savefig("plot2_lev_profile.png", dpi=150)
    plt.show()
    print("Saved -> plot2_lev_profile.png")


def plot_lev_time(variables, titles):
    if time_vals is not None:
        try:
            import cftime
            t0 = time_vals[0]
            t_axis = np.array([
                cftime.date2num(t, units=f"days since {t0.year}-{t0.month:02d}-{t0.day:02d}",
                                calendar=t0.calendar)
                for t in time_vals
            ])
            t_label = f"Days since {t0.year}-{t0.month:02d}-{t0.day:02d}"
        except Exception:
            t_axis = np.array(time_vals, dtype="float64")
            t_label = "Time"
    else:
        t_axis = np.arange(next(iter(variables.values())).shape[0])
        t_label = "Time index"
    lat_mask = (lat >= LAT_MIN) & (lat <= LAT_MAX)
    fig, axes = _make_grid(len(variables))
    for ax, (key, arr) in zip(axes, variables.items()):
        data    = np.nanmean(arr[:, :, lat_mask], axis=2)
        vmax_ov = LEV_TIME_VMAX.get(key)
        cf      = ax.contourf(t_axis, lev, data.T, **_contourf_kwargs(data, vmax_override=vmax_ov))
        ax.contour(t_axis, lev, data.T, levels=cf.levels[::2], colors="k", linewidths=0.4, alpha=0.5)
        _add_colorbar(fig, cf, ax)
        _format_lev_ax(ax, t_label)
        ax.set_title(titles[key], fontsize=10)
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right", fontsize=7)
    for ax in axes[len(variables):]:
        ax.set_visible(False)
    fig.suptitle("Lev x Time  (lat-mean)", fontsize=13, fontweight="bold")
    fig.savefig("plot3_lev_time.png", dpi=150)
    plt.show()
    print("Saved -> plot3_lev_time.png")


plot_lev_lat(VARIABLES, TITLES)
plot_lev_profile(VARIABLES, TITLES)
plot_lev_time(VARIABLES, TITLES)
